# Section 4 — Model Deployment (FastAPI + Streaming)
Serves Qwen2.5-1.5B-Instruct as a REST API with streaming.

**Runtime:** GPU (T4) required

In [ ]:
!pip install -q fastapi uvicorn transformers torch accelerate bitsandbytes httpx pyngrok

In [ ]:
import torch, time, asyncio, json, os
from threading import Thread
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
import uvicorn

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

# Load 4-bit model
print('Loading model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map='auto', trust_remote_code=True)
model.eval()
print(f'Model loaded. VRAM: {torch.cuda.memory_allocated()/1024**2:.0f} MB')

In [ ]:
# === FastAPI App ===
app = FastAPI(title='Qwen2.5 API')

class GenRequest(BaseModel):
    prompt: str
    max_tokens: int = 256
    temperature: float = 0.7

class GenResponse(BaseModel):
    text: str
    tokens_generated: int
    total_time_ms: float
    tokens_per_second: float

@app.get('/health')
def health():
    return {'status': 'healthy', 'model': MODEL_ID, 'device': 'cuda', 'quantized': True,
            'vram_mb': torch.cuda.memory_allocated()/1024**2}

@app.post('/generate', response_model=GenResponse)
def generate(req: GenRequest):
    msgs = [{'role': 'user', 'content': req.prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=req.max_tokens,
                            temperature=req.temperature if req.temperature > 0 else 1.0,
                            do_sample=req.temperature > 0)
    elapsed = time.perf_counter() - t0
    new_toks = out[0][inputs['input_ids'].shape[1]:]
    return GenResponse(text=tokenizer.decode(new_toks, skip_special_tokens=True),
                       tokens_generated=len(new_toks), total_time_ms=elapsed*1000,
                       tokens_per_second=len(new_toks)/elapsed)

@app.post('/generate/stream')
def generate_stream(req: GenRequest):
    msgs = [{'role': 'user', 'content': req.prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    kwargs = {**inputs, 'max_new_tokens': req.max_tokens, 'streamer': streamer,
              'temperature': req.temperature if req.temperature > 0 else 1.0,
              'do_sample': req.temperature > 0}
    Thread(target=lambda: model.generate(**kwargs)).start()
    def gen():
        for tok in streamer:
            if tok:
                yield f'data: {tok}\n\n'
        yield 'data: [DONE]\n\n'
    return StreamingResponse(gen(), media_type='text/event-stream')

print('FastAPI app defined.')

In [ ]:
# === Start server in background ===
import nest_asyncio
nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000)

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print('Server started on port 8000')

In [ ]:
# === Load Test ===
import httpx

BASE = 'http://localhost:8000'
PROMPT = 'Explain machine learning in 3 sentences.'
N = 10

# Health check
r = httpx.get(f'{BASE}/health')
print(f'Health: {r.json()}')

# Streaming TTFT test
print('\n--- Streaming TTFT Test ---')
t0 = time.perf_counter()
ttft = None
tokens = 0
with httpx.stream('POST', f'{BASE}/generate/stream', json={'prompt': PROMPT, 'max_tokens': 100}, timeout=60) as resp:
    for line in resp.iter_lines():
        if line.startswith('data: ') and not line.startswith('data: [DONE]'):
            if ttft is None:
                ttft = time.perf_counter() - t0
            tokens += 1
total = time.perf_counter() - t0
print(f'TTFT: {ttft*1000:.0f} ms | Total: {total*1000:.0f} ms | Tokens: {tokens}')

# Concurrent test
print(f'\n--- {N} Concurrent Requests ---')
async def concurrent_test():
    async with httpx.AsyncClient(timeout=60) as client:
        async def req(i):
            t = time.perf_counter()
            r = await client.post(f'{BASE}/generate', json={'prompt': PROMPT, 'max_tokens': 80})
            return {'id': i, 'ms': (time.perf_counter()-t)*1000, 'ok': r.status_code==200,
                    'tps': r.json().get('tokens_per_second',0) if r.status_code==200 else 0}
        t0 = time.perf_counter()
        results = await asyncio.gather(*[req(i) for i in range(N)])
        wall = time.perf_counter() - t0
    return results, wall

results, wall = await concurrent_test()
ok = [r for r in results if r['ok']]
lats = [r['ms'] for r in ok]
print(f'Success: {len(ok)}/{N}')
if lats:
    print(f'Avg latency: {sum(lats)/len(lats):.0f} ms')
    print(f'Min: {min(lats):.0f} ms | Max: {max(lats):.0f} ms')
    print(f'P50: {sorted(lats)[len(lats)//2]:.0f} ms')
    print(f'Wall-clock (all {N}): {wall*1000:.0f} ms')
    print(f'Avg tok/s: {sum(r["tps"] for r in ok)/len(ok):.1f}')

In [ ]:
# === Dockerfile (for reference) ===
dockerfile = '''
FROM nvidia/cuda:12.1.0-runtime-ubuntu22.04
RUN apt-get update && apt-get install -y python3.11 python3-pip && rm -rf /var/lib/apt/lists/*
WORKDIR /app
COPY requirements.txt .
RUN pip3 install --no-cache-dir -r requirements.txt
COPY app.py .
ENV MODEL_ID=Qwen/Qwen2.5-1.5B-Instruct USE_4BIT=true PORT=8000
EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=10s CMD python3 -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"
CMD ["python3", "app.py"]
'''
print('Dockerfile:')
print(dockerfile)
print('\nTo build and run:')
print('  docker build -t llm-api .')
print('  docker run --gpus all -p 8000:8000 llm-api')